In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
# load the dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# preprocessing the data
# dropping the irrelevant columns/features
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# applying the encoding for categorical variables
label_encoder_gender = LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data['Gender']


0       0
1       0
2       0
3       0
4       0
       ..
9995    1
9996    1
9997    0
9998    1
9999    0
Name: Gender, Length: 10000, dtype: int32

In [5]:
print(data['Geography'].unique())
print(data['Gender'].unique())

<ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str
[0 1]


In [6]:
# use one hot encoding for the geography col
from sklearn.preprocessing import OneHotEncoder
# ohe = OneHotEncoder(sparse=False)
onehot_encoder_geo = OneHotEncoder()
geo_encoding = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoding

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
geo_encoding.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoded_df = pd.DataFrame(geo_encoding.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [11]:
# combining the encoded cols with the original data

data = pd.concat( [data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [12]:
data.columns
data.info()
data.isnull().sum()
data.describe()
data.size

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        10000 non-null  int64  
 1   Gender             10000 non-null  int32  
 2   Age                10000 non-null  int64  
 3   Tenure             10000 non-null  int64  
 4   Balance            10000 non-null  float64
 5   NumOfProducts      10000 non-null  int64  
 6   HasCrCard          10000 non-null  int64  
 7   IsActiveMember     10000 non-null  int64  
 8   EstimatedSalary    10000 non-null  float64
 9   Exited             10000 non-null  int64  
 10  Geography_France   10000 non-null  float64
 11  Geography_Germany  10000 non-null  float64
 12  Geography_Spain    10000 non-null  float64
dtypes: float64(5), int32(1), int64(7)
memory usage: 976.7 KB


130000

In [13]:
# save the enoders  and scaler 
with open('label_encoder_gender.pkl','wb') as file:
   pickle.dump(label_encoder_gender,file) 

with open('onehot_encoder_geo.pkl','wb') as file:
   pickle.dump(onehot_encoder_geo,file)



In [14]:
# divde the dataset into independent and independent features 
X=data.drop('Exited',axis=1)
y=data['Exited']

# split the data in tarining and test sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# scaling down the features using the standard scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [15]:
print(X_train)
print(X_test)
print(X_train.shape,
X_test.shape,
y_train.shape,
y_test.shape)

[[ 0.35649971  0.91324755 -0.6557859  ...  1.00150113 -0.57946723
  -0.57638802]
 [-0.20389777  0.91324755  0.29493847 ... -0.99850112  1.72572313
  -0.57638802]
 [-0.96147213  0.91324755 -1.41636539 ... -0.99850112 -0.57946723
   1.73494238]
 ...
 [ 0.86500853 -1.09499335 -0.08535128 ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.15932282  0.91324755  0.3900109  ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.47065475  0.91324755  1.15059039 ... -0.99850112  1.72572313
  -0.57638802]]
[[-0.57749609  0.91324755 -0.6557859  ... -0.99850112  1.72572313
  -0.57638802]
 [-0.29729735  0.91324755  0.3900109  ...  1.00150113 -0.57946723
  -0.57638802]
 [-0.52560743 -1.09499335  0.48508334 ... -0.99850112 -0.57946723
   1.73494238]
 ...
 [ 0.81311987 -1.09499335  0.77030065 ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.41876609  0.91324755 -0.94100321 ...  1.00150113 -0.57946723
  -0.57638802]
 [-0.24540869  0.91324755  0.00972116 ... -0.99850112  1.72572313
  -0.57638802]]
(8000, 12) (2000

In [16]:
with open('scaler.pkl','wb') as file:
   pickle.dump(scaler,file)

# Ann implemention

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [18]:
print(X_train.shape[1])
print(X_train[1].shape,)

12
(12,)


In [19]:
# building the network/ANN model
model = Sequential([
   Dense(64,activation='relu',input_shape= (X_train.shape[1],)), #HL1 connnected to input laye
   Dense(32,activation='relu'), #HL2
   Dense(1,activation='sigmoid') #output layer
])


In [20]:
graph = tf.compat.v1.get_default_graph()
print(graph)

In [21]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
import tensorflow 
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
# more optimizers = [Adadelta,Adafactor,Adagrad,Adam,Adamw,sgd,rmsprop]
losses = tensorflow.keras.losses.BinaryCrossentropy()
met = tensorflow.keras.metrics.Accuracy()

In [23]:
# model.compile(optimizer=opt,loss=losses,metrics=met)
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

In [24]:
# set up the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callbacks = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [25]:
# set up for early stopping
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [26]:
# train the model
history =model.fit(
   X_train,y_train,validation_data =(X_test,y_test),epochs=100,
   callbacks=[tensorflow_callbacks,early_stopping_callback]
)

Epoch 1/100

250/250 [==============================] - 8s 12ms/step - loss: 0.4600 - accuracy: 0.7987 - val_loss: 0.3988 - val_accuracy: 0.8300
Epoch 2/100
250/250 [==============================] - 3s 11ms/step - loss: 0.3936 - accuracy: 0.8349 - val_loss: 0.3637 - val_accuracy: 0.8510
Epoch 3/100
250/250 [==============================] - 2s 8ms/step - loss: 0.3616 - accuracy: 0.8549 - val_loss: 0.3454 - val_accuracy: 0.8570
Epoch 4/100
250/250 [==============================] - 2s 9ms/step - loss: 0.3465 - accuracy: 0.8593 - val_loss: 0.3428 - val_accuracy: 0.8620
Epoch 5/100
250/250 [==============================] - 2s 8ms/step - loss: 0.3399 - accuracy: 0.8601 - val_loss: 0.3442 - val_accuracy: 0.8625
Epoch 6/100
250/250 [==============================] - 2s 8ms/step - loss: 0.3355 - accuracy: 0.8625 - val_loss: 0.3458 - val_accuracy: 0.8590
Epoch 7/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3335 - accuracy: 0.8602 - val_loss: 0.3412 - val_accuracy: 0.8

In [27]:
model.save('model.h5')

c:\Users\Shaheen\anaconda3\envs\tf-env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [4]:
# load the tensorboard extension
%load_ext tensorboard


In [2]:
!pip install tensorboard

  Using cached tensorboard-2.21.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached grpcio-1.82.1-cp311-cp311-win_amd64.whl.metadata (3.8 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.3.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
Using cached tensorboard-2.21.0-py3-none-any.whl (5.5 MB)
Using cached grpcio-1.82.1-cp311-cp311-win_amd64.whl (5.0 MB)
Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl (439 kB)
Using cached tensorboard_data_server-0.7.2-py3-none-any.whl (2.4 kB)
Using cached absl_py-2.5.0-py3-non

In [10]:
# magic functions
%tensorboard --logdir logs/fit20260717-232609

Reusing TensorBoard on port 6007 (pid 16012), started 0:00:37 ago. (Use '!kill 16012' to kill it.)

In [ ]:
# load the pickle file


In [7]:
%tensorboard --logdir logs/fit20260717-232609